In [1]:
# %pip install dotenv gradio_client httpx uvicorn scikit-learn nest_asyncio

In [2]:
import os
import json
import re
import asyncio
from typing import List, Dict, Any, Tuple
from pathlib import Path
from datetime import datetime
from fastapi.middleware.cors import CORSMiddleware
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk
import heapq
import httpx
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gradio_client import Client as GradioClient

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

app = FastAPI(title="Empathia ADHD Copilot API")

# Add CORS middleware to allow frontend to communicate
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:3001"],  # Next.js dev server
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─── Colab Gradio Connection ───
# Paste the Gradio share URL from Colab here (e.g. https://xxxxx.gradio.live)
COLAB_GRADIO_URL = os.environ.get("COLAB_GRADIO_URL", "https://c98b85aeef052c3333.gradio.live")
print(f"\u2728 Colab Gradio URL: {COLAB_GRADIO_URL}")
print("Connecting to Colab Gradio endpoint...")
try:
    gradio_llm = GradioClient(COLAB_GRADIO_URL)
    print("\u2705 Connected to Colab LLM successfully!")
except Exception as e:
    print(f"\u26a0\ufe0f Could not connect to Colab: {e}")
    print("Make sure the Colab notebook is running and the Gradio URL is correct.")
    gradio_llm = None

# Fine-tuned NLP model server endpoint (local cognitive shift scoring)
NLP_SERVER_URL = "http://localhost:8001"
print(f"NLP Server: {NLP_SERVER_URL}")

SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive": "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":    "taking action, making decisions, expressing control, planning, choosing"
}

# ─────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────

class ChatRequest(BaseModel):
    session_id: int
    message: str
    history: List[Dict[str, str]]

# ─────────────────────────────────────────────
# LOGIC HELPERS & RAG
# ─────────────────────────────────────────────

async def score_text(text: str):
    """
    Call the fine-tuned model server to score text.
    Returns dict with scores (0-1) for each category and dominant category.
    """
    try:
        async with httpx.AsyncClient() as client_http:
            response = await client_http.post(
                f"{NLP_SERVER_URL}/score",
                json={"text": text},
                timeout=10.0
            )
            response.raise_for_status()
            return response.json()
    except Exception as e:
        print(f"Error calling NLP server: {e}")
        # Graceful fallback
        return {"scores": {"affective": 0, "cognitive": 0, "agency": 0}, "dominant": "unknown"}

def build_rag_corpus():
    documents = []
    for path in sorted(SESSIONS_DIR.glob("session_*.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                session_id = data.get("session_id", "Unknown")
                for msg in data.get("conversation", []):
                    content = msg.get("content", "")
                    notes = msg.get("notes", {})
                    if isinstance(notes, dict) and "notes" in notes:
                        for note in notes["notes"]:
                            content += f" | NOTE ({note.get('category')}): {note.get('title')} - {note.get('content')}"
                    if content.strip():
                        documents.append({
                            "session": session_id,
                            "role": msg.get("role", "user").capitalize(),
                            "text": content,
                            "timestamp": msg.get("timestamp", "")[:10]
                        })
        except Exception as e:
            print(f"Warning: could not read {path} for RAG: {e}")
    return documents

def get_relevant_context(current_message: str, top_k: int = 3):
    docs = build_rag_corpus()
    if not docs:
        return "No past session history available."
    
    texts = [d["text"] for d in docs]
    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        tfidf_matrix = vectorizer.fit_transform(texts)
        query_vec = vectorizer.transform([current_message])
        similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
        
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        relevant_blocks = []
        for idx in top_indices:
            # threshold score so we don't pull completely unrelated things
            if similarities[idx] > 0.05:
                doc = docs[idx]
                relevant_blocks.append(f"[Session {doc['session']} - {doc['timestamp']}] {doc['role']}: {doc['text']}")
                
        if not relevant_blocks:
             return "No highly relevant past context found for this specific query."
        
        return "RELEVANT PAST CONTEXT:\n" + "\n".join(relevant_blocks)
    except Exception as e:
        print(f"RAG Error: {e}")
        return "Error retrieving history."

def load_or_create_session(session_id: int) -> dict:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    else:
        return {
            "session_id": session_id,
            "created": datetime.now().isoformat(),
            "conversation": [],
            "cognitive_shifts": []
        }

def save_session(session_id: int, session_data: dict) -> None:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w") as f:
        json.dump(session_data, f, indent=2)

def extract_notes_from_response(raw_reply: str) -> Tuple[str, Dict[str, Any]]:
    notes: Dict[str, Any] = {}
    clean_text = raw_reply
    if "### UPDATED_NOTES" in raw_reply:
        clean_text = raw_reply.split("### UPDATED_NOTES")[0].strip()
        json_part = raw_reply.split("### UPDATED_NOTES")[-1].strip()
        try:
            json_part = re.sub(r"^```(?:json)?", "", json_part).strip()
            json_part = re.sub(r"```$", "", json_part).strip()
            json_match = re.search(r'\{.*\}', json_part, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                notes = json.loads(json_str)
            else:
                notes = json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse JSON notes: {e}")
            notes = {"error": "Failed to parse notes", "raw": json_part[:200]}
    else:
        try:
            json_match = re.search(r'\{[^{}]*\"[^\"]*\"[^{}]*\}', raw_reply, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                parsed = json.loads(json_str)
                clean_text = raw_reply[:json_match.start()].strip()
                notes = parsed
        except:
            pass
    return clean_text, notes

# ─────────────────────────────────────────────
# ENDPOINTS
# ─────────────────────────────────────────────

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        # 1. Cognitive Shift Analysis (call NLP server)
        analysis = await score_text(req.message)
        
        # 2. Get RAG Context (Semantic Search using TF-IDF)
        past_context = get_relevant_context(req.message)
        
        # 3. Construct System Prompt
        system_prompt = f"""You are Indy, an empathetic, multilingual therapeutic assistant. You support code-switching (e.g., Hindi-English, Spanish-English) and always provide non-judgmental, validating responses.\n\nHere is some semantically relevant context pulled from past sessions for this user. You may use this context to inform your responses.\n\n{past_context}\n"""

        # 4. Call fine-tuned LLM on Colab via Gradio API
        messages = [{"role": "system", "content": system_prompt}] + req.history + [{"role": "user", "content": req.message}]
        
        try:
            if gradio_llm is None:
                raise Exception("Gradio client not connected. Restart the cell after setting COLAB_GRADIO_URL.")
            
            # Call Colab Gradio API (run in executor since gradio_client is sync)
            loop = asyncio.get_event_loop()
            raw_reply = await loop.run_in_executor(
                None,
                lambda: gradio_llm.predict(
                    json.dumps(messages),  # messages as JSON string
                    1024,                  # max_new_tokens
                    0.7,                   # temperature
                    api_name="/generate"
                )
            )
        except Exception as llm_error:
            print(f"Colab Gradio API Error: {llm_error}")
            raise HTTPException(status_code=502, detail=f"Colab LLM error. Check if Colab notebook is running and Gradio URL is correct: {str(llm_error)}")
        
        # 5. Parsing & Cleaning
        clean_text = raw_reply
        
        # Generate summary of user's message using NLTK logic
        user_summary = summarize_text(req.message, num_sentences=1)
        
        notes = {
            "session_id": req.session_id,
            "date": datetime.now().strftime("%Y-%m-%d"),
            "session_type": "brain_dump",
            "notes": [
              {
                "title": f"{datetime.now().strftime('%I:%M:%S %p')}",
                "content": user_summary,
                "category": "auto_summary"
              }
            ]
        }
        
        

        # 6. Save session with conversation history
        session_data = load_or_create_session(req.session_id)
        session_data["conversation"].append({
            "role": "user",
            "content": req.message,
            "timestamp": datetime.now().isoformat()
        })
        session_data["conversation"].append({
            "role": "assistant",
            "content": clean_text,
            "timestamp": datetime.now().isoformat(),
            "cognitive_shift": analysis,
            "notes": notes
        })
        session_data["cognitive_shifts"].append({
            "message": req.message,
            "analysis": analysis,
            "timestamp": datetime.now().isoformat()
        })
        
        save_session(req.session_id, session_data)
        
        return {
            "reply": clean_text,
            "cognitive_shift": analysis,
            "extracted_notes": notes
        }

    except HTTPException:
        raise
    except Exception as e:
        print(f"Error in chat endpoint: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import heapq

# Prepare NLTK datasets (will download quietly)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

def summarize_text(text, num_sentences=1):
    if not text.strip():
        return ""
        
    try:
        stop_words = set(stopwords.words('english'))
        words = word_tokenize(text.lower())
        
        word_frequencies = {}
        for word in words:
            if word.isalnum() and word not in stop_words:
                word_frequencies[word] = word_frequencies.get(word, 0) + 1

        if not word_frequencies:
            return text

        max_freq = max(word_frequencies.values())
        for word in word_frequencies.keys():
            word_frequencies[word] = word_frequencies[word] / max_freq

        sentences = sent_tokenize(text)
        if len(sentences) <= num_sentences:
            return text
            
        sentence_scores = {}
        for sent in sentences:
            for word in word_tokenize(sent.lower()):
                if word in word_frequencies:
                    if sent not in sentence_scores:
                        sentence_scores[sent] = word_frequencies[word]
                    else:
                        sentence_scores[sent] += word_frequencies[word]

        summary_sentences = heapq.nlargest(num_sentences, sentence_scores, key=sentence_scores.get)
        return ' '.join(summary_sentences)
    except Exception as e:
        print(f"Summarization error: {e}")
        return text

c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✨ Colab Gradio URL: https://c98b85aeef052c3333.gradio.live
Connecting to Colab Gradio endpoint...
Loaded as API: https://c98b85aeef052c3333.gradio.live/ ✔
⚠️ Could not connect to Colab: Could not fetch api info for https://c98b85aeef052c3333.gradio.live/: {"detail":"Not Found"}
Make sure the Colab notebook is running and the Gradio URL is correct.
NLP Server: http://localhost:8001


In [3]:

@app.get("/sessions/{session_id}/cognitive-shifts")
async def get_cognitive_shifts(session_id: int):
    """
    Retrieve cognitive shift data for all user messages in a session.
    Returns an array of shifts with message number, scores, and dominant category.
    """
    try:
        session_data = load_or_create_session(session_id)
        
        # Extract cognitive shifts from conversation history
        # Only include shifts for user messages
        shifts = []
        message_num = 0
        
        for i, msg in enumerate(session_data["conversation"]):
            if msg["role"] == "user":
                message_num += 1
                # Get the corresponding assistant response (should be next message)
                if i + 1 < len(session_data["conversation"]):
                    assistant_msg = session_data["conversation"][i + 1]
                    if "cognitive_shift" in assistant_msg:
                        shift_data = assistant_msg["cognitive_shift"]
                        shifts.append({
                            "message_num": message_num,
                            "affective": shift_data["scores"].get("affective", 0),
                            "cognitive": shift_data["scores"].get("cognitive", 0),
                            "agency": shift_data["scores"].get("agency", 0),
                            "dominant": shift_data.get("dominant", ""),
                            "timestamp": msg.get("timestamp", ""),
                            "content": msg.get("content", "")[:100]  # First 100 chars
                        })
        
        return {
            "session_id": session_id,
            "total_messages": message_num,
            "shifts": shifts
        }
    
    except Exception as e:
        print(f"Error retrieving cognitive shifts: {e}")
        raise HTTPException(status_code=500, detail=str(e))

In [4]:
import json

def merge_notes(existing_notes: dict, llm_response: str) -> dict:
    """
    Parses the UPDATED_NOTES JSON from the LLM response and merges it
    with the existing notes. The LLM is expected to return the full
    merged object, so this function simply validates and returns it.
    Falls back to existing notes if parsing fails.
    """
    marker = "### UPDATED_NOTES"
    if marker not in llm_response:
        return existing_notes

    raw_json = llm_response.split(marker, 1)[1].strip()

    try:
        updated = json.loads(raw_json)
        # Ensure the new notes dict is merged on top of existing notes
        # so no old keys are silently dropped if the LLM forgets them
        if "notes" in existing_notes and "notes" in updated:
            merged_notes = {**existing_notes.get("notes", {}), **updated["notes"]}
            updated["notes"] = merged_notes
        return updated
    except json.JSONDecodeError:
        # If the LLM returned malformed JSON, preserve what we had
        return existing_notes

In [ ]:
# Start the server
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def _start_uvicorn():
	# Run uvicorn in a separate thread so we don't call asyncio.run()
	# from within Jupyter's running event loop.
	uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=_start_uvicorn, daemon=True)
thread.start()

print("Uvicorn server started in background thread on port 8000")

Uvicorn server started in background thread on port 8000


INFO:     Started server process [33136]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
